# GDS trap with a patterned buried ground — 5 µm spacer

Import the patterned ground from `Zhi_75um_PatternedGNDPLANE_FINAL.gds`
into COMSOL 6.1. **RF domain 37 is held at 1 V; all other electrodes are at 0 V.** No stationary solve is run.

| Region | z range (µm) |
|---|---:|
| Top metal, GDS **7/0** | **0 to 0.5** |
| Oxide above buried metal | **−5 to 0** |
| Buried GND, GDS **8/0** | **−5.5 to −5** |
| Oxide refill where buried metal is absent | **−5.5 to −5** |
| Lower oxide, 12 µm | **−17.5 to −5.5** |
| Silicon, 675 µm | **−692.5 to −17.5** |

There is **5 µm of oxide above the buried metal**, and **5.5 µm of oxide
where that metal is absent**, above the separate 12 µm lower oxide.
Both GDS layers keep their original xy coordinates.

Run in order using a fresh kernel after geometry changes. COMSOL needs the
ECAD Import Module; Python needs `MPh` and `numpy`. The final cell builds
and saves the complete mesh by default. RF assignment is optional.

In [1]:
from pathlib import Path
import numpy as np
import mph
from jpype import JArray
from jpype.types import JBoolean, JInt

# If needed: %pip install MPh numpy
GDS_FILE = Path(r"C:\Users\Administrator\Downloads\Zhi_75um_PatternedGNDPLANE_FINAL.gds")
GDS_CELL = 'COMSOL_RF_1000UM_NO_VIAS'  # Cell name as read by COMSOL 6.1.

# Stack dimensions [um], measured from the underside of the top electrodes.
TOP_THICKNESS = 0.5
GND_THICKNESS = 0.5
OXIDE_ABOVE_GND = 5.0
LOWER_OXIDE_THICKNESS = 12.0
SILICON_THICKNESS = 675.0
Z_GND_TOP = -OXIDE_ABOVE_GND
Z_GND_BOTTOM = Z_GND_TOP - GND_THICKNESS
Z_LOWER_BOTTOM = Z_GND_BOTTOM - LOWER_OXIDE_THICKNESS
Z_SILICON_BOTTOM = Z_LOWER_BOTTOM - SILICON_THICKNESS

CHIP_MARGIN = 0.0        # Added around the top-mask bounding box [um].
AIR_PADDING = 250.0      # Lateral clearance around the chip [um].
AIR_BOTTOM, AIR_TOP = -1000.0, 250.0

# Mesh lengths [um] and elements through the thin dielectric layers.
SURFACE_SIZE = 80.0
BURIED_SIZE = 40.0
MIN_SIZE = 0.2
OXIDE_LAYERS = 6         # Through the upper 5 um; six sweep layers (at least five).
REFILL_LAYERS = 2        # Through the 0.5 um beside buried metal.
LOWER_OXIDE_LAYERS = 6   # Through the lower 12 um.
AIR_GAP_LAYERS = 2       # Through the 0.5 um air beside top metal.
BUILD_MESH = True       # False saves only the geometry and mesh instructions.


## 1. Import 7/0 and 8/0

Each Import enables one exact layer/datatype pair, with manual elevation
and thickness. COMSOL 6.1 exposes these as `LAYER7_0` and `LAYER8_0` when
datatype splitting is enabled. The imports create named domain selections.

In [2]:
client = mph.start(version='6.1', cores=4)
model = client.create('Patterned buried GND - 5 um oxide')
j = model.java
j.component().create('comp1', JBoolean(True))
comp = j.component('comp1')
comp.label('Trap, patterned ground, oxide and silicon')
comp.geom().create('geom1', 3)
geom = comp.geom('geom1')
geom.lengthUnit('um')
geom.label('Layer stack [um]; top-electrode underside at z = 0')

def import_layer(tag, label, layer, z, thickness):
    feature = geom.create(tag, 'Import')
    feature.label(label)
    feature.set('filename', str(GDS_FILE))
    feature.set('updategeomunit', JBoolean(False))
    feature.set('cell', GDS_CELL)
    feature.set('splitbydatatype', 'on')
    feature.set('grouping', 'layer')
    feature.set('importtype', 'full3d')
    feature.set('manualelevation', 'on')
    feature.set('findarcs', 'off')
    feature.set('selresult', 'on')
    feature.set('selresultshow', 'all')
    names = [str(row[0]) for row in feature.getStringMatrix('layerprop')]
    if layer not in names:
        raise ValueError(f'{layer} is absent from the GDS: {names}')
    feature.set('importlayer', ['on' if name == layer else 'off' for name in names])
    feature.set('height', [f'{thickness}[um]' if name == layer else '0[um]' for name in names])
    feature.set('elevation', [f'{z}[um]' if name == layer else '0[um]' for name in names])
    feature.importData()
    geom.run(tag)
    return feature

top = import_layer('top', 'Top metal 7/0 | z = 0 to 0.5 um', 'LAYER7_0', 0.0, TOP_THICKNESS)
ground = import_layer('buried', 'Patterned GND 8/0 | z = -5.5 to -5 um',
                      'LAYER8_0', Z_GND_BOTTOM, GND_THICKNESS)

measure = geom.measure()
measure.selection().set(list(top.objectNames()))
xmin, xmax, ymin, ymax, _, _ = np.asarray(measure.getBoundingBox(), dtype=float)
xmin, xmax = xmin-CHIP_MARGIN, xmax+CHIP_MARGIN
ymin, ymax = ymin-CHIP_MARGIN, ymax+CHIP_MARGIN
chip_x, chip_y = xmax-xmin, ymax-ymin
print(f'Chip: x = {xmin:g} to {xmax:g}, y = {ymin:g} to {ymax:g} um')


Chip: x = -5500 to 6000, y = -3250 to 3250 um


## 2. Build the oxide, silicon, and air

The chip footprint follows the top mask. The spacer is split only where
needed for sweeping: upper oxide (−5 to 0) and oxide refill (−5.5 to −5).
Importing the 8/0 footprint once more through the upper oxide partitions
it into an inside and outside region; **both are SiO₂**. This gives the
upper-oxide sweep compatible source and destination faces.

One thin air slab surrounds the top electrodes at z = 0 to 0.5 µm. Its
swept mesh resolves the metal sidewalls without forcing the bulk air to use
very small tetrahedra there. The remaining air encloses the full chip; no
side-air rings are added. Differences retain their subtracting objects.

In [3]:
def block(tag, label, xy, z, thickness):
    x, y, width, depth = xy
    feature = geom.create(tag, 'Block')
    feature.label(label)
    feature.set('base', 'corner')
    feature.set('pos', [str(x), str(y), str(z)])
    feature.set('size', [str(width), str(depth), str(thickness)])
    feature.set('selresult', 'on')
    feature.set('selresultshow', 'all')

def difference(tag, label, whole, subtract):
    feature = geom.create(tag, 'Difference')
    feature.label(label)
    feature.selection('input').set([whole])
    feature.selection('input2').set(subtract)
    feature.set('keepsubtract', 'on')
    feature.set('intbnd', 'on')
    feature.set('selresult', 'on')
    feature.set('selresultshow', 'all')

chip_xy = [xmin, ymin, chip_x, chip_y]
import_layer('oxide_inside', 'SiO2 above GND | z = -5 to 0 um',
             'LAYER8_0', Z_GND_TOP, OXIDE_ABOVE_GND)
block('upper_box', 'Upper spacer envelope', chip_xy, Z_GND_TOP, OXIDE_ABOVE_GND)
difference('oxide_outside', 'SiO2 above GND openings | z = -5 to 0 um', 'upper_box', ['oxide_inside'])

block('refill_box', 'Buried-level envelope', chip_xy, Z_GND_BOTTOM, GND_THICKNESS)
difference('refill', 'SiO2 in GND openings | z = -5.5 to -5 um', 'refill_box', ['buried'])
block('lower_oxide', 'Lower SiO2 | z = -17.5 to -5.5 um',
      chip_xy, Z_LOWER_BOTTOM, LOWER_OXIDE_THICKNESS)
block('silicon', 'Silicon | z = -692.5 to -17.5 um',
      chip_xy, Z_SILICON_BOTTOM, SILICON_THICKNESS)

air_xy = [xmin-AIR_PADDING, ymin-AIR_PADDING,
          chip_x+2*AIR_PADDING, chip_y+2*AIR_PADDING]
block('gap_box', 'Air envelope at top-metal height', air_xy, 0.0, TOP_THICKNESS)
difference('air_gap', 'Air beside top metal | z = 0 to 0.5 um', 'gap_box', ['top'])
block('air_box', 'Air enclosure', air_xy, AIR_BOTTOM, AIR_TOP-AIR_BOTTOM)
difference('air', 'Air around the complete chip', 'air_box',
           ['top', 'buried', 'oxide_inside', 'oxide_outside',
            'refill', 'lower_oxide', 'silicon', 'air_gap'])
geom.feature('fin').set('action', 'union')
geom.run()
print('Geometry built.')


Geometry built.


## 3. Domain selections and your RF choice

The table below lists the top-metal domain numbers and their xy bounding
boxes. Use it, or inspect domain labels in the saved COMSOL model, to choose
RF. Set **`RF_DOMAIN`** in the following cell. Leaving it `None` builds and
saves the geometry and mesh setup without assigning electrostatics.

The buried GND selection comes directly from 8/0. No RF recognition or
geometric matching is performed.

In [4]:
def domains(tag):
    return [int(v) for v in comp.selection(f'geom1_{tag}_dom').entities(JInt(3))]

def boundaries(domain_ids):
    return sorted({int(b) for d in domain_ids
                   for b in geom.getAdj(JInt(3), JInt(2), JInt(d))})

def bounding_box(dimension, entity):
    measure = comp.measure()
    measure.selection().geom('geom1', JInt(dimension))
    measure.selection().set(JArray(JInt)([entity]))
    return np.asarray(measure.getBoundingBox(), dtype=float)

def horizontal_faces(domain_ids, z):
    return [b for b in boundaries(domain_ids)
            if np.allclose(bounding_box(2, b)[4:], [z, z], rtol=0, atol=1e-6)]

def select(selection, dimension, entities):
    selection.geom('geom1', JInt(dimension))
    selection.set(JArray(JInt)(list(entities)))

def named_selection(tag, label, dimension, entities):
    if tag in [str(t) for t in comp.selection().tags()]:
        selection = comp.selection(tag)
    else:
        selection = comp.selection().create(tag, 'Explicit')
    selection.label(label)
    select(selection, dimension, entities)

top_ids = domains('top')
buried_ids = domains('buried')
upper_ids = domains('oxide_inside') + domains('oxide_outside')
refill_ids = domains('refill')
lower_ids = domains('lower_oxide')
silicon_ids = domains('silicon')
gap_ids = domains('air_gap')
bulk_air_ids = domains('air')
air_ids = gap_ids + bulk_air_ids
oxide_ids = upper_ids + refill_ids + lower_ids
dielectric_ids = oxide_ids + silicon_ids + air_ids

for tag, label, ids in [('TOP_METAL', 'Metal | top electrodes 7/0', top_ids),
                        ('BURIED_GND', 'Metal | patterned buried ground 8/0', buried_ids),
                        ('SIO2', 'All SiO2, including buried-level refill', oxide_ids),
                        ('SILICON', 'Silicon', silicon_ids),
                        ('AIR', 'Air', air_ids),
                        ('DIELECTRICS', 'All dielectric domains', dielectric_ids)]:
    named_selection(tag, label, 3, ids)

print('Top-metal domains:   xmin      xmax      ymin      ymax  [um]')
for domain in top_ids:
    bounds = bounding_box(3, domain)[:4]
    print(f'{domain:18d}: ' + ' '.join(f'{value:9.2f}' for value in bounds))
print('Buried GND domains:', buried_ids)


Top-metal domains:   xmin      xmax      ymin      ymax  [um]
                 8:  -5500.00   6000.00  -3250.00   3250.00
                12:  -5200.00   5396.00    -38.50   2950.00
                22:  -4722.00   -594.13    131.50   2950.00
                26:  -4244.00   -474.13    131.50   2950.00
                28:  -3766.00   -354.13    131.50   2950.00
                31:  -3288.00   -234.13    131.50   2950.00
                32:  -2810.00   -114.13    131.50   2950.00
                33:  -2332.00    105.87    131.50   2950.00
                36:  -1854.00    325.87    131.50   2950.00
                37:  -1706.13   5700.00   -138.50    848.50
                38:  -1376.00    445.87    131.50   2950.00
                39:   -898.00    665.87    131.50   2950.00
                40:   -706.13   1398.00  -2950.00   -146.50
                41:   -586.13   1876.00  -2950.00   -146.50
                42:   -466.13   2354.00  -2950.00   -146.50
                43:   -420.00    885.8

In [5]:
RF_DOMAIN = 37  # User-selected RF electrode; terminal voltage is 1 V.

## 4. Materials and optional electrostatics

Air, SiO₂, and dielectric silicon use εᵣ = 1, 3.9, and 11.7, respectively.
If you selected RF, it is held at 1 V; all other top metal and all buried
metal are grounded. Metal interiors are meshed, but remain excluded from
Electrostatics; conductor potentials are imposed on their boundaries. To change RF, edit the previous cell and rerun this cell and
the save cell. The stationary study remains unsolved.

In [6]:
for tag, label, epsr, selection in [('mat_air', 'Air', 1.0, 'AIR'),
                                   ('mat_oxide', 'SiO2', 3.9, 'SIO2'),
                                   ('mat_si', 'Silicon', 11.7, 'SILICON')]:
    if tag not in [str(t) for t in comp.material().tags()]:
        material = comp.material().create(tag, 'Common')
        material.label(label)
        material.selection().named(selection)
        material.propertyGroup('def').set('relpermittivity',
            [str(epsr), '0', '0', '0', str(epsr), '0', '0', '0', str(epsr)])

if 'es' in [str(t) for t in comp.physics().tags()]:
    comp.physics().remove('es')
if RF_DOMAIN is None:
    print('RF is not selected. Geometry and mesh can still be built and saved.')
else:
    if RF_DOMAIN not in top_ids:
        raise ValueError('RF_DOMAIN must be one of the top-metal domain numbers above.')
    ground_ids = [d for d in top_ids if d != RF_DOMAIN] + buried_ids
    named_selection('RF', 'RF boundaries (user selected)', 2, boundaries([RF_DOMAIN]))
    named_selection('GND', 'Other top electrodes and buried GND', 2, boundaries(ground_ids))
    j.param().set('Vrf', '1[V]')
    es = comp.physics().create('es', 'Electrostatics', 'geom1')
    es.label('Electrostatics | dielectric domains only')
    es.selection().named('DIELECTRICS')
    es.create('gnd1', 'Ground', JInt(2))
    es.feature('gnd1').label('Ground | all other electrodes, 0 V')
    es.feature('gnd1').selection().named('GND')
    es.create('term1', 'Terminal', JInt(2))
    es.feature('term1').label(f'RF | domain {RF_DOMAIN}, 1 V')
    es.feature('term1').selection().named('RF')
    es.feature('term1').set('TerminalType', 'Voltage')
    es.feature('term1').set('V0', 'Vrf')
    if 'std1' not in [str(t) for t in j.study().tags()]:
        j.study().create('std1')
        j.study('std1').label('Capacitance | stationary, not yet solved')
        j.study('std1').create('stat', 'Stationary')
    print(f'RF = domain {RF_DOMAIN}; electrostatics configured, not solved.')


RF = domain 37; electrostatics configured, not solved.


## 5. Mesh every domain

Triangulate the upper spacer at z = 0 and sweep downward through 5 µm in six layers.
Sweep both the oxide refill and buried metal from −5 to −5.5 µm, using the
same thickness divisions on their shared sidewalls. These sweeps supply
the complete source mesh for the 12 µm lower oxide.

At the top level, sweep both air and metal from 0 to 0.5 µm with matching
thickness divisions. Silicon and bulk air use tetrahedra. No geometry
objects are added for the metal meshes, and every solid now receives
volume elements. Metal interiors remain outside Electrostatics.

The upper-oxide partition projects the buried footprint onto the sweep
source. See COMSOL's [swept-mesh guidance](https://www.comsol.com/support/learning-center/article/how-to-control-the-swept-mesh-50881)
for source/destination requirements. Sizes are starting values; mesh and
air-box convergence are still needed for capacitance.

The mesh is explicitly **User-controlled**. Every sweep has source and
destination faces selected by height; no automatic face detection is needed.
Keep this sequence and use **Build All** in COMSOL.

In [7]:
if 'mesh1' in [str(tag) for tag in comp.mesh().tags()]:
    comp.mesh().remove('mesh1')
comp.mesh().create('mesh1')
mesh = comp.mesh('mesh1')
mesh.label('Complete mesh | explicit sweeps, then bulk tetrahedra')

def mesh_size(parent, dimension, entities, maximum, tag='size1'):
    size = parent.create(tag, 'Size')
    select(size.selection(), dimension, entities)
    size.set('custom', JBoolean(True))
    size.set('hmax', f'{maximum}[um]')
    size.set('hmin', f'{MIN_SIZE}[um]')
    size.set('hgrad', '1.5')

def triangles(tag, label, faces, maximum):
    feature = mesh.create(tag, 'FreeTri')
    feature.label(label)
    select(feature.selection(), 2, faces)
    mesh_size(feature, 2, faces, maximum)

def sweep(tag, label, ids, source_z, target_z, layers):
    feature = mesh.create(tag, 'Sweep')
    feature.label(f'{label} | z = {source_z:g} to {target_z:g} um')
    select(feature.selection(), 3, ids)
    select(feature.selection('sourceface'), 2, horizontal_faces(ids, source_z))
    select(feature.selection('targetface'), 2, horizontal_faces(ids, target_z))
    feature.set('facemethod', 'tri')
    feature.set('sweeppath', 'straight')
    distribution = feature.create('dist1', 'Distribution')
    select(distribution.selection(), 3, ids)
    distribution.label(f'{layers} elements through thickness')
    distribution.set('numelem', JInt(layers))

triangles('tri_top', 'Top spacer surface', horizontal_faces(upper_ids, 0), SURFACE_SIZE)
mesh_size(mesh.feature('tri_top'), 2, horizontal_faces(domains('oxide_inside'), 0),
          BURIED_SIZE, tag='size_buried')
sweep('sweep_upper', 'Upper oxide', upper_ids, 0, Z_GND_TOP, OXIDE_LAYERS)
sweep('sweep_refill', '500 nm oxide beside buried GND', refill_ids, Z_GND_TOP, Z_GND_BOTTOM, REFILL_LAYERS)
sweep('sweep_buried', '500 nm buried GND metal', buried_ids, Z_GND_TOP, Z_GND_BOTTOM, REFILL_LAYERS)
sweep('sweep_lower', 'Lower 12 um oxide', lower_ids, Z_GND_BOTTOM, Z_LOWER_BOTTOM, LOWER_OXIDE_LAYERS)

# The part of the air-gap source inside the chip is already meshed.
outside_faces = sorted(set(horizontal_faces(gap_ids, 0))
                       - set(horizontal_faces(upper_ids, 0)))
triangles('tri_gap', 'Air-gap source outside chip', outside_faces, SURFACE_SIZE)
sweep('sweep_gap', 'Air beside 500 nm top metal', gap_ids, 0, TOP_THICKNESS, AIR_GAP_LAYERS)
sweep('sweep_top_metal', '500 nm top electrode metal', top_ids, 0, TOP_THICKNESS, AIR_GAP_LAYERS)

for tag, label, ids, maximum in [('tet_silicon', 'Silicon', silicon_ids, 400.0),
                                ('tet_air', 'Bulk air around chip', bulk_air_ids, 600.0)]:
    feature = mesh.create(tag, 'FreeTet')
    feature.label(label)
    select(feature.selection(), 3, ids)
    mesh_size(feature, 3, ids, maximum)
# Adding explicit operations makes the sequence user-controlled.
mesh.automatic(JBoolean(False))
print('Mesh sequence configured.')


Mesh sequence configured.


## 6. Save the setup

The `Zhi75um_oxide5um_RF37_1V` filenames keep earlier models and results separate. You can
open the setup in COMSOL to inspect the domain numbers before choosing RF.
Rerun this cell after making that choice. No mesh or study is run by saving.

The `_setup.mph` file has mesh instructions but no elements. For a built
mesh, run the next section and open `_meshed.mph`. In COMSOL, use **Complete mesh > Build All** (User-controlled) to execute the entire sequence, rather than a selected feature.

In [8]:
output_dir = Path.cwd()
for parent in (Path.cwd(), *Path.cwd().parents):
    candidate = parent / 'Simulations' / 'trap_capacitance'
    if candidate.is_dir():
        output_dir = candidate
        break
setup_file = output_dir / 'COMSOL_Zhi75um_oxide5um_RF37_1V_setup.mph'
mesh_file = output_dir / 'COMSOL_Zhi75um_oxide5um_RF37_1V_meshed.mph'
model.save(str(setup_file))
print('Saved:', setup_file)


Saved: c:\Users\Administrator\Desktop\Yiyang_GITHUB\integrated_trapped_ions\Simulations\trap_capacitance\COMSOL_Zhi75um_oxide5um_RF37_1V_setup.mph


## 7. Build and save the complete mesh

`BUILD_MESH=True` is the default. Run this cell to build the mesh. It builds every mesh stage and
checks actual element coverage of all geometry domains, including both
metal layers. The separate `_meshed.mph` file is saved only after that
coverage check passes. No stationary solve is run.

The bulk-air mesher can report low-quality elements; inspect those messages
before solving. Quality is reported separately for tetrahedra, prisms, and
transition pyramids. Complete domain coverage is not a convergence check.

In [9]:
if BUILD_MESH:
    stages = ['sweep_upper', 'sweep_refill', 'sweep_buried', 'sweep_lower',
              'sweep_gap', 'sweep_top_metal', 'tet_silicon', 'tet_air']
    for tag in stages:
        print(f'Building {tag} ...', flush=True)
        mesh.run(tag)
    meshed_domains = {int(d) for kind in ('tet', 'prism', 'pyr')
                      for d in mesh.getGeomEntities(kind)}
    missing = set(range(1, int(geom.getNDomains())+1)) - meshed_domains
    if missing:
        raise RuntimeError(f'Domains without volume elements: {sorted(missing)}')
    print(f'All {geom.getNDomains()} domains have volume elements.')
    print('Elements:', int(mesh.getNumElem()))
    for kind in ('tet', 'prism', 'pyr'):
        print(f'{kind}: {mesh.getNumElem(kind)} elements; '
              f'minimum quality {mesh.getMinQuality(kind):.3g}')
    for mesh_node in (model / 'meshes').children():
        for problem in mesh_node.problems():
            print('Mesh message:', problem['message'])
    model.save(str(mesh_file))
    print('Saved meshed model:', mesh_file)
else:
    print('Mesh not built. Set BUILD_MESH=True when ready.')


Building sweep_upper ...
Building sweep_refill ...
Building sweep_buried ...
Building sweep_lower ...
Building sweep_gap ...
Building sweep_top_metal ...
Building tet_silicon ...
Building tet_air ...
All 67 domains have volume elements.
Elements: 6319265
tet: 2606969 elements; minimum quality 3.75e-06
prism: 3699990 elements; minimum quality 0.00488
pyr: 12306 elements; minimum quality 3.11e-06
Mesh message: Generated 2908 elements of low quality. The worst quality is 0.0004611. Positions of elements with lowest quality.
Saved meshed model: c:\Users\Administrator\Desktop\Yiyang_GITHUB\integrated_trapped_ions\Simulations\trap_capacitance\COMSOL_Zhi75um_oxide5um_RF37_1V_meshed.mph
